# Pitch Arsenal Analysis

**Objective:** Compare each pitcher's arsenal in detail - velocity, movement, spin, and swing-and-miss effectiveness.

**Key Question:** Does having more pitch types (Ohtani has 7) actually translate to better results than a focused 3-pitch mix (Sanchez throws SI/CH/SL)?

In [ ]:
from analysis_utils import load_all, set_style
import pandas as pd, numpy as np, matplotlib.pyplot as plt

set_style()
dfs = load_all()
COLORS = {'Ohtani':'#1f77b4','Sanchez':'#ff7f0e','Misiorowski':'#2ca02c'}

In [ ]:
# ---- Detailed Arsenal Comparison Table ----
rows = []
for name, df in dfs.items():
    for pt in sorted(df['pitch_type'].dropna().unique()):
        sub = df[df['pitch_type']==pt]
        n = len(sub)
        if n < 5: continue
        vel = sub['release_speed'].dropna()
        rows.append({
            'Pitcher': name, 'Pitch': pt, 'n': n, 'Usage%': round(n/len(df)*100,1),
            'Velo': round(vel.mean(),1), 'MaxVelo': round(vel.max(),1),
            'Spin': round(sub['release_spin_rate'].mean(),0),
            'HMov': round(sub['pfx_x'].mean(),1),
            'VMov': round(sub['pfx_z'].mean(),1),
            'Extension': round(sub['release_extension'].mean(),2),
        })
tbl = pd.DataFrame(rows)
print("ARSENAL COMPARISON (pitches used >= 5 times)")
print("=" * 130)
print(tbl.to_string(index=False))
tbl.to_csv('figures/arsenal_table.csv', index=False)

In [ ]:
# ---- Whiff Rate by Pitch Type ----
print("WHIFF RATES BY PITCH TYPE\n" + "="*70)
data = []
for name, df in dfs.items():
    for pt in sorted(df['pitch_type'].dropna().unique()):
        sub = df[df['pitch_type']==pt]
        swings = sub[sub['description'].str.contains(
            'swinging_strike|hit_into_play|foul', na=False, regex=True)]
        whiffs = sub[sub['description'].str.contains('swinging_strike', na=False)]
        n_sw = len(swings); n_w = len(whiffs)
        wp = n_w/n_sw*100 if n_sw > 0 else 0
        data.append({'Pitcher':name,'Pitch':pt,'Swings':n_sw,'Whiffs':n_w,'Whiff%':f'{wp:.1f}%'})
print(pd.DataFrame(data).to_string(index=False))

print("\n\nOVERALL WHIFF RATE")
for name, df in dfs.items():
    swings = df[df['description'].str.contains(
        'swinging_strike|hit_into_play|foul', na=False, regex=True)]
    whiffs = df[df['description'].str.contains('swinging_strike', na=False)]
    wp = len(whiffs)/len(swings)*100 if len(swings)>0 else 0
    print(f"  {name:12s}  {len(whiffs):3d}/{len(swings):3d}  =  {wp:.1f}%")

In [ ]:
# ---- Velocity Comparison ----
fig, ax = plt.subplots(figsize=(12, 6))
pitch_order = ['FF','SI','FC','SL','ST','CU','CH','FS']
visible = [p for p in pitch_order if any(p in dfs[n]['pitch_type'].dropna().unique() for n in dfs)]
offsets = {'Ohtani':-0.2,'Sanchez':0,'Misiorowski':0.2}
markers = {'Ohtani':'o','Sanchez':'s','Misiorowski':'D'}
for idx, pt in enumerate(visible):
    for name in dfs:
        vel = dfs[name][dfs[name]['pitch_type']==pt]['release_speed'].dropna()
        if len(vel) < 5: continue
        x = idx + offsets[name]
        ax.errorbar(x, vel.mean(), yerr=vel.std(), fmt=markers[name],
                   color=COLORS[name], capsize=4, capthick=2, markersize=10,
                   label=name if idx==0 else '')
ax.set_xticks(range(len(visible))); ax.set_xticklabels(visible)
ax.set_ylabel("Velocity (mph)")
ax.set_title("Velocity by Pitch Type (Mean +/- SD)", fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('figures/velocity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Spin Rate Comparison ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, (name, df) in zip(axes, dfs.items()):
    pts = sorted(df['pitch_type'].dropna().unique())
    data = [df[df['pitch_type']==pt]['release_spin_rate'].dropna().values for pt in pts]
    bp = ax.boxplot(data, label=pts, patch_artist=True)
    for patch, c in zip(bp['boxes'], plt.cm.Set2(np.linspace(0,1,len(pts)))):
        patch.set_facecolor(c)
    ax.set_title(name, fontweight='bold')
    ax.set_ylabel("Spin Rate (rpm)")
plt.suptitle("Spin Rate Distribution by Pitch Type", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/spin_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Key Findings ----
print("""
KEY FINDINGS:
1. Sanchez throws only 3 pitch types, yet his changeup (42.0% whiff) and slider (42.7% whiff)
   are MORE effective swing-and-miss weapons than Ohtani's best secondary (ST at 37.1%).
2. Ohtani's broad arsenal may indicate he is searching for an out-pitch rather than having one.
3. Misiorowski's four-seamer (100.5 mph, 37.2% whiff) is the single most dominant pitch among the three.
4. Ohtani's overall whiff rate (28.4%) ranks LAST among the three pitchers.
""")